In [1]:
# Importing libraries
import torch

import json
import matplotlib.pyplot as plt
plt.rc('xtick', labelsize=18)    # fontsize of the tick labels
plt.rc('ytick', labelsize=18)    # fontsize of the tick labels
import numpy as np
from nltk.corpus import wordnet as wn
#from numpy import random
import os
import requests
import time
from dataclasses import dataclass, asdict, field
from time import strftime, gmtime
datetag = strftime("%Y-%m-%d", gmtime())
data_cache = 'cached_data'
HOST, device = os.uname()[1], torch.device("cuda" if torch.cuda.is_available() else "cpu")

    
# to store results
import pandas as pd

@dataclass
class Params:
    DEBUG = 1
    
    datetag: str = datetag # Set the date of the result's file
    loader: str = 'cached_data/Imagenet_urls_ILSVRC_2016.json' # File containing Imagenet's labels
    data_root: str = '/data/JNJER/Deep_learning/data/Imagenet_square'
    root: str = '/data/JNJER/Deep_learning/data/ILSVRC/Data/CLS-LOC/' # Directory containing images to perform the training
    
    
    folders: list = field(default_factory=lambda: ['val', 'train']) # Set the training and validation folders relative to the root

args = Params()



#to plot & display 
def pprint(message): #display function
    print('\n')
    print('-'*len(message))
    print(message)
    print('-'*len(message))

data_ = {}

    

with open(args.loader) as json_file:
    Imagenet_urls_ILSVRC_2016 = json.load(json_file)


# gathering labels
labels = []
for i_img, img_id in enumerate(Imagenet_urls_ILSVRC_2016):
    syn_= wn.synset_from_pos_and_offset('n', int(img_id.split(' ')[0][1:]))
    labels.append(syn_.lemmas()[0].name())
    
    
def get_boxes(df, value):
    idx = list(df['ImageId'][df['ImageId'] == value].index)
    box = []
    if idx:
        for i in range(len(df["PredictionString"][idx[0]].split(' '))//5):
            pos =(5 *i)
            box.append({'xmin' : df["PredictionString"][idx[0]].split(' ')[1 + pos],
                                                             'ymin' : df["PredictionString"][idx[0]].split(' ')[2 + pos],
                                                         'xmax' : df["PredictionString"][idx[0]].split(' ')[3 + (5 *i)],
                                                     'ymax' : df["PredictionString"][idx[0]].split(' ')[4 + (5 *i)]})
    return box

def clean_list(list_dir, patterns=['.DS_Store', '.ipynb_checkpoints']):
    for pattern in patterns:
        if pattern in list_dir: list_dir.remove(pattern)
    return list_dir

In [ ]:
import shutil
from PIL import Image, ImageDraw, ImageFont # the Pillow module (PIL) is supported by default by TorchVision

verbose = False


df_data = {}

def square_box(xmin, ymin, xmax, ymax):
    temp = ((xmax-xmin)-(ymax-ymin))//2
    if temp > 0 :
        ymin -= temp
        ymax += temp
    else:
        xmin += temp
        xmax -= temp
    return xmin, ymin, xmax, ymax


for folder in args.folders :
    with open(f'cached_data/LOC_{folder}_solution.csv', 'r') as csv_file:
        df_data[folder] = pd.read_csv(csv_file)


#for folder in args.folders :
for folder in ['train'] :
    print(f'\nFolder \"{folder}\"')
    os.makedirs(folder, exist_ok=True)
    source_folder = os.path.join(args.root, folder)
    boxes_folder = os.path.join(args.data_root, folder)
    os.makedirs(boxes_folder, exist_ok=True)
    for i_img, img_id in enumerate(Imagenet_urls_ILSVRC_2016):
        print(f'\nScraping images for id \"{img_id}\" : {labels[i_img]} ')
        target_folder = os.path.join(boxes_folder, img_id) 
        os.makedirs(target_folder, exist_ok=True)
        final_source_folder = os.path.join(source_folder, img_id)
        for imgs in  clean_list(os.listdir(final_source_folder)):
            data_local = os.path.join(final_source_folder, imgs)
            obj = get_boxes(df_data[folder], imgs.split('.')[0])

            original_image = Image.open(data_local, mode='r')
            if len(obj)  > 0 :
                try:
                    xmin = int(obj[0]['xmin'])
                    ymin = int(obj[0]['ymin'])
                    xmax = int(obj[0]['xmax'])
                    ymax = int(obj[0]['ymax'])

                    crop_image = original_image.crop(square_box(xmin, ymin, xmax, ymax))
                    crop_image.save(os.path.join(target_folder, imgs))
                except:
                    pass

            print(f'\r{len(clean_list(os.listdir(target_folder)))} / {len(clean_list(os.listdir(final_source_folder)))}', end="", flush=not verbose)



Folder "train"

Scraping images for id "n01440764" : tench 
454 / 1300
Scraping images for id "n01443537" : goldfish 
648 / 1300
Scraping images for id "n01484850" : great_white_shark 
530 / 1300
Scraping images for id "n01491361" : tiger_shark 
485 / 1300
Scraping images for id "n01494475" : hammerhead 
472 / 1300
Scraping images for id "n01496331" : electric_ray 
465 / 1300
Scraping images for id "n01498041" : stingray 
501 / 1300
Scraping images for id "n01514668" : cock 
584 / 1300
Scraping images for id "n01514859" : hen 
495 / 1300
Scraping images for id "n01518878" : ostrich 
512 / 1300
Scraping images for id "n01530575" : brambling 
430 / 1300
Scraping images for id "n01531178" : goldfinch 
563 / 1300
Scraping images for id "n01532829" : house_finch 
591 / 1300
Scraping images for id "n01534433" : junco 
531 / 1300
Scraping images for id "n01537544" : indigo_bunting 
488 / 1300
Scraping images for id "n01558993" : robin 
518 / 1300
Scraping images for id "n01560419" : bulbul 
